In [1]:
# Install required ML libraries
%pip install scikit-learn mlflow

StatementMeta(, 26b2fc4d-0718-4b45-8df7-0231ea0e2b3e, 8, Finished, Available, Finished, False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 204.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 165.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 171.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 224.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# ── Read Silver data ──────────────────────────────────────────────────────────
df     = spark.read.format("delta").table("lh_silver.dbo.capacity_utilization").toPandas()
df_inv = spark.read.format("delta").table("lh_silver.dbo.server_inventory").toPandas()

# ── Feature engineering ───────────────────────────────────────────────────────
features = ["cpu_pct", "ram_pct", "storage_pct", "network_mbps", "power_watts"]
X = df[features].apply(pd.to_numeric, errors="coerce").dropna()

# ── Standardize features ──────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Train Isolation Forest ────────────────────────────────────────────────────
# contamination=0.1 means we expect ~10% anomalous readings
model = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
model.fit(X_scaled)

# ── Score all records ─────────────────────────────────────────────────────────
df_scored = df.loc[X.index, ["server_id", "timestamp"]].copy().reset_index(drop=True)
df_scored["cpu_pct"]       = X["cpu_pct"].values
df_scored["ram_pct"]       = X["ram_pct"].values
df_scored["storage_pct"]   = X["storage_pct"].values
df_scored["network_mbps"]  = X["network_mbps"].values
df_scored["power_watts"]   = X["power_watts"].values
df_scored["anomaly_score"] = np.round(model.decision_function(X_scaled), 4)
df_scored["anomaly_flag"]  = model.predict(X_scaled)
df_scored["is_anomaly"]    = df_scored["anomaly_flag"].apply(
    lambda x: "Anomaly" if x == -1 else "Normal"
)

# ── Add risk level ────────────────────────────────────────────────────────────
df_scored["risk_level"] = pd.cut(
    df_scored["anomaly_score"],
    bins=[-np.inf, -0.1, 0.0, np.inf],
    labels=["High Risk", "Medium Risk", "Low Risk"]
).astype(str)

# ── Join with server inventory for location context ───────────────────────────
df_final = df_scored.merge(
    df_inv[["server_id", "data_center_location", "server_type", "status"]],
    on="server_id", how="left"
)

# ── Log to MLflow experiment ──────────────────────────────────────────────────
mlflow.set_experiment("DataCenter360_AnomalyDetection")
with mlflow.start_run():
    mlflow.log_params({
        "n_estimators":  100,
        "contamination": 0.1,
        "algorithm":     "IsolationForest",
        "features":      str(features)
    })
    mlflow.log_metrics({
        "anomaly_rate":       round(len(df_final[df_final["is_anomaly"]=="Anomaly"]) / len(df_final), 4),
        "total_records":      len(df_final),
        "anomalies_detected": int(len(df_final[df_final["is_anomaly"]=="Anomaly"])),
        "high_risk_servers":  int(len(df_final[df_final["risk_level"]=="High Risk"])),
        "medium_risk_servers":int(len(df_final[df_final["risk_level"]=="Medium Risk"]))
    })
    print("✅ MLflow experiment logged successfully")

# ── Drop existing table to avoid schema conflicts ─────────────────────────────
spark.sql("DROP TABLE IF EXISTS server_anomaly_scores")

# ── Write to Gold with correct numeric types ──────────────────────────────────
spark.createDataFrame(df_final).write.format("delta").mode("overwrite").saveAsTable("server_anomaly_scores")

print(f"✅ server_anomaly_scores: {len(df_final)} rows written to lh_gold")
print(f"\n   Anomaly breakdown:")
print(f"   High Risk:    {len(df_final[df_final['risk_level']=='High Risk'])}")
print(f"   Medium Risk:  {len(df_final[df_final['risk_level']=='Medium Risk'])}")
print(f"   Low Risk:     {len(df_final[df_final['risk_level']=='Low Risk'])}")
print(f"\nColumn dtypes:")
print(df_final.dtypes)

StatementMeta(, 032fac1c-9eac-4099-a139-152906cdebae, 4, Finished, Available, Finished, False)

✅ MLflow experiment logged successfully
✅ server_anomaly_scores: 151 rows written to lh_gold

   Anomaly breakdown:
   High Risk:    3
   Medium Risk:  13
   Low Risk:     135

Column dtypes:
server_id                       object
timestamp               datetime64[us]
cpu_pct                        float32
ram_pct                        float32
storage_pct                    float32
network_mbps                   float32
power_watts                    float32
anomaly_score                  float64
anomaly_flag                     int64
is_anomaly                      object
risk_level                      object
data_center_location            object
server_type                     object
status                          object
dtype: object


2026-04-24:21:12:40,755 ERROR    [shared_platform_utils.py:45] list MLExperiment failed, status_code: 500, b'{"Message":"An error has occurred."}'
2026-04-24:21:12:40,755 ERROR    [synapse_mlflow_utils.py:390] [fabric mlflow plugin]: <class 'synapse.ml.mlflow.tracking_store.TridentMLflowTrackingStore'>.check_experiment_artifact_id exception (500, <synapse.ml.mlflow.model.shared_artifact.ErrorResponse object at 0x706957f8ec90>)


In [3]:
# Export updated anomaly scores to CSV for Streamlit app
df_final.to_csv("/lakehouse/default/Files/server_anomaly_scores.csv", index=False)
print(f"✅ server_anomaly_scores.csv exported to lh_gold/Files/")

StatementMeta(, 032fac1c-9eac-4099-a139-152906cdebae, 5, Finished, Available, Finished, False)

✅ server_anomaly_scores.csv exported to lh_gold/Files/
